# 02. Cross-Entropy & Classification Loss Functions

**The mathematical derivation and intuition behind Binary Cross-Entropy (BCE), Categorical Cross-Entropy (CCE), and why Softmax + Cross-Entropy is the standard for AI classification.**

---

## 1. What is Cross-Entropy?

Suppose the **true distribution** of data is $P(x)$ (ground-truth one-hot label), but our machine learning model predicts an **estimated distribution** $Q(x)$ (softmax output probabilities).

**Cross-Entropy** measures the average number of bits needed to encode events from distribution $P$ using the model's predicted distribution $Q$:

$$H(P, Q) = -\sum_x P(x) \log Q(x)$$

- When $Q(x) = P(x)$ (the model predicts ground truth perfectly), $H(P, Q) = H(P)$ (minimum possible entropy).
- When $Q(x)$ deviates from $P(x)$, $H(P, Q)$ increases dramatically, heavily penalizing wrong, confident predictions!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Comparing Cross-Entropy Loss vs Mean Squared Error (MSE) for Classification
# True label y = 1, Predicted probability p ranges from 0.01 to 0.99
p = np.linspace(0.01, 0.99, 200)

ce_loss = -np.log(p) # Cross Entropy when y = 1
mse_loss = (1.0 - p)**2 # MSE when y = 1

plt.figure(figsize=(9, 5))
plt.plot(p, ce_loss, 'r-', linewidth=2.5, label='Cross-Entropy Loss: -log(p)')
plt.plot(p, mse_loss, 'b--', linewidth=2, label='MSE Loss: (1 - p)^2')
plt.title("Why Cross-Entropy is Superior for Classification: Massive Penalty when p -> 0")
plt.xlabel("Predicted Probability for True Class p")
plt.ylabel("Loss Value")
plt.ylim(0, 5)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---

## 2. Binary Cross-Entropy (BCE / Log-Loss)

For binary classification where $y \in \{0, 1\}$ and the model predicts probability $\hat{y} = \sigma(z) \in [0, 1]$:

$$\mathcal{L}_{BCE}(y, \hat{y}) = - \left[ y \log \hat{y} + (1 - y) \log (1 - \hat{y}) \right]$$

- If true $y = 1$: $\mathcal{L} = -\log \hat{y}$
- If true $y = 0$: $\mathcal{L} = -\log (1 - \hat{y})$


In [ ]:
def binary_cross_entropy(y_true, y_pred):
    eps = 1e-15 # numerical stability clipping
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

y_true = np.array([1, 0, 1, 1, 0])
y_good_pred = np.array([0.95, 0.05, 0.88, 0.92, 0.08])
y_bad_pred  = np.array([0.10, 0.80, 0.20, 0.05, 0.90])

print("BCE for Accurate Model:", binary_cross_entropy(y_true, y_good_pred))
print("BCE for Poor Model:    ", binary_cross_entropy(y_true, y_bad_pred))


---

## 3. Categorical Cross-Entropy (CCE) & Softmax

For multi-class classification with $K$ classes where $\mathbf{y}$ is a one-hot vector ($y_c = 1$ for true class $c$) and $\mathbf{\hat{y}} = \text{Softmax}(\mathbf{z})$:

$$\mathcal{L}_{CCE} = -\sum_{k=1}^K y_k \log \hat{y}_k = -\log \hat{y}_c \quad \text{where } \hat{y}_k = \frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}}$$

### The Mathematical Magic: Clean Gradient Derivation
Why is Softmax combined with Cross-Entropy in every neural network?
Let's take the derivative of $\mathcal{L}_{CCE}$ with respect to the raw logit $z_i$:

$$\frac{\partial \mathcal{L}_{CCE}}{\partial z_i} = \hat{y}_i - y_i$$

### Why this is incredible:
- The gradient is simply **(Predicted Probability - True Label)**!
- No complex non-linear terms, no gradient saturation, perfectly stable linear error signal during backpropagation!


In [ ]:
def categorical_cross_entropy(y_true_one_hot, logits):
    # Log-sum-exp trick for numerical stability
    log_sum_exp = np.log(np.sum(np.exp(logits - np.max(logits, axis=-1, keepdims=True)), axis=-1, keepdims=True)) + np.max(logits, axis=-1, keepdims=True)
    log_probs = logits - log_sum_exp
    loss = -np.sum(y_true_one_hot * log_probs)
    
    # Softmax probabilities
    probs = np.exp(log_probs)
    # Gradient: p - y
    grad = probs - y_true_one_hot
    return loss, probs, grad

logits = np.array([2.5, 1.0, 0.2]) # Model predictions for 3 classes
y_true = np.array([1.0, 0.0, 0.0]) # True class is Class 0

loss, probs, grad = categorical_cross_entropy(y_true, logits)
print(f"Logits: {logits}")
print(f"Softmax Probabilities: {np.round(probs, 4)}")
print(f"CCE Loss: {loss:.4f}")
print(f"Gradient dL/dz (Probs - y_true): {np.round(grad, 4)}")


---

## 4. Summary & Key Takeaways

1. **Cross-Entropy** $H(P, Q) = -\sum P(x) \log Q(x)$ quantifies how well model predictions match true probability distributions.
2. For binary classification, it is **Binary Cross-Entropy (BCE)**; for multi-class, it is **Categorical Cross-Entropy (CCE)**.
3. Combining Softmax with Cross-Entropy produces the remarkably elegant and numerically stable gradient:
   $$rac{\partial \mathcal{L}}{\partial \mathbf{z}} = \mathbf{\hat{y}} - \mathbf{y}$$
